# 2.0.0 Kidney Replacement Calculator Demonstration

In version 2.0.0 of the dashboard stats calculator the algorithms to calculate the incident and prevalent cohorts have been fully overhauled. 

In [1]:
from rr_connection_manager import PostgresConnection

conn = PostgresConnection(app = "ukrdc_staging", tunnel = True, via_app = True)
conn.connection_check()
sessionmaker = conn.session_maker()


Connection to ukrdc_staging successful. 
Database info: 
	PostgreSQL 9.6.24 on x86_64-pc-linux-gnu


# Running Calculator 

The basic syntax is the same with all the calculators. The KRT calculator has a period of time implicit for the calculations of the incident cohort. For the backtesting this was set to a year to allow granular comparison with the cohorts in the annual report.

In [6]:
import json
import datetime as dt
from ukrdc_stats.calculators.krt import KRTStatsCalculator


facility = "RFBAK"
start = dt.datetime(2021, 12, 31)
end = dt.datetime(2022, 12, 31)

#start = dt.datetime(2024, 1, 1)
#end = dt.datetime.now()


with sessionmaker() as session:
    calculator = KRTStatsCalculator(session=session, facility=facility, from_time=start, to_time=end)
    output = calculator.extract_stats()


formatted_output = json.dumps(
    json.loads(
        output.all.json()
    ), 
    indent=4
)

print(formatted_output)


{
    "all_treatments_krt": {
        "metadata": {
            "title": "All KRT Modalities",
            "summary": "Breakdown of all patients on both PD and HD, and by home therapies and in-centre therapies.",
            "description": "\n# All Patients Undergoing Kidney Replacement Therapy\n\n## Overview\nThis pie chart illustrates the proportion of patients who received kidney replacement therapy within the time period. The chart is broken down by the type of treatment, including HD In-center, HD Home, HD Unknown/Incomplete, PD, and Tx. Optionally the chart can be filtered by satellite unit. \n\n## Treatment Definitions\n- HD: Haemodialysis patients (with a modality defined as HD by the UKRDC). This includes patients registered for haemodialysis, haemofiltration, haemodiafiltration, or ultrafiltration. \n- PD: Peritoneal dialysis (with a modality defined as PD by the UKRDC).This includes patients registered for CAPD or APD treatments.\n- TX: Transplant patients (with a modality d

In [7]:
from IPython.display import display, Markdown
import plotly.express as px


display(Markdown(output.all.prevalent_krt.metadata.description))
units = ", ".join(output.units.keys())

display(
    Markdown(
f"""
### Total Patient Population : {output.all.prevalent_krt.metadata.population_size}
### Satellite Units : {units}
"""
    )
)



all_patients = px.pie(
    names = output.all.prevalent_krt.data.x,
    values = output.all.prevalent_krt.data.y,
    hole=0.6,
)
all_patients.show()
#all_patients.show("png")




# Prevalent Kidney Replacement Patients

## Definition
A patient receiving ongoing kidney replacement therapy (KRT) - defined as haemodialysis, peritoneal dialysis, 
or kidney transplant - who is established on treatment at the end of the selected time period.

## Inclusion Criteria
1. Treatment continues beyond end of selected dates
2. Has received at least 90 days of treatment before window end
3. No gaps in treatment greater than 90 days
4. Either:
    - Active treatment at window end
    - Recent transfer to another unit (discharge code 38)

## Timeline Example 
```
Example Cases:
X = Treatment
- = No Treatment
* = Death
T = Transfer

                                                Prevalence Point                                        
|<-------------------->|<--------------------------->|
                        [Analysis Window]         [Must be on treatment]

Prevalent        XXXXXX|XXXXXXXXXXXXXXXXXXXXXXXXXXXXX|X----  (Active at end)
Prevalent        XXXXXX|XXXXXXXXXXXXXXXXXXXXXXXXT--->|>----  (Transfer out)
Prevalent        --XXXX|XXXXXXXX------------XXXXXXXXX|X----  (>90 days at end)
Prevalent        ------|----------------------------X|XXXXX  (>90 days at end)

Not Prevalent    XXXXXX|XXXXXXXXXXXXXXX*-------------|-----  (Died in window)
Not Prevalent    ------|----------------------XXXX---|X----  (<90 days total)
```    

## UKRDC Entities Used
The chart was produced by joining the following UKRDC entities according to their foreign key relationships:
- [PatientRecord](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450149/PatientRecord): ukrdcid, sendingextract
- [Patient](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450145/Patient): deathtime
- [Treatment](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450155/Treatment+Encounter): qbl05, hdp04, fromtime, totime, dischargereasoncode, healthcarefacilitycode
- [ModalityCodes](https://renalregistry.atlassian.net/l/cp/Ac1YeFfH): registry_code_type

## Data Quality & Limitations
The software used to calculate the statistics should be considered experimental and is subject to the following non-exhaustive limitations:

1. Data Coverage:
- Patients from another unit may be counted as prevalent if they are being treated in the unit at prevalence point

2. Data Quality:
- UKRDC contains uncleaned data and may be incomplete or contain errors




### Total Patient Population : 1285
### Satellite Units : RNQ51, RFBAT, RFBAK, RY5K7, RGN, 9RP7LA, RT5DC, RKZDA, 9RWDLB, 98RFBAK, 9RWD, 995


In [4]:
display(Markdown(output.all.incentre_dialysis_frequency.metadata.description))
freq_fig = px.bar(
    x=output.all.incentre_dialysis_frequency.data.y,
    y=output.all.incentre_dialysis_frequency.data.x,
    title=output.all.incentre_dialysis_frequency.metadata.title,
    labels={
        "x": output.all.incentre_dialysis_frequency.metadata.axis_titles.y,
        "y": output.all.incentre_dialysis_frequency.metadata.axis_titles.x,
    },
    orientation="h",
    color_discrete_sequence=["rgb(243,159,33)"],
    text_auto=True,
)
freq_fig.show()


# In-Centre Dialysis Frequency

## Overview
This histogram represents the mean number of dialysis sessions per week for all dialysis patients in a three month period at a sendingfacility or one of its satellites. Optionally the chart can be filtered by satellite unit. 

## Methodology
- Dialysis sessions are counted for patients in the 'All Patients Undergoing Kidney Replacement Therapy' cohort. This is done by grouping on the procedure type code. 
- Patients with less than two sessions are rejected. 
- The per week frequency is calculated for each person by dividing the count by the time difference between their first and last dialysis session within the three month period.
- Patients are aggregated into bins of with boundaries (0.0, 0.5, 1.5, 2.5, 3.5, 7.0). This are labelled 1,2,3 and >3 sessions per week.  

## UKRDC Entities Used
The dialysis sessions table is queried by grouping by ukrdcid with the following aggregate functions used:
- https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2005565449/Dialysis+Session+Procedure: MIN(fromtime), MAX(totime), COUNT(sessiontype).


In [5]:
display(Markdown(output.all.incident_krt.metadata.description))

display(
    Markdown(
f"""
### Total Patient Population : {output.all.incident_krt.metadata.population_size}
"""
    )
)



all_patients = px.pie(
    names = output.all.incident_krt.data.x,
    values = output.all.incident_krt.data.y,
    hole=0.6,
)
all_patients.show()


# Incident Kidney Replacement Patients

## Definition
A patient starting kidney replacement therapy (KRT) - defined as haemodialysis, peritoneal dialysis, 
or kidney transplant - for the first time during the selected time period.

## Inclusion Criteria
1. First treatment starts within the selected dates. This is defined as the first treatment preceded by a gap of greater than 90 days without KRT treatment.
2. Either:
   - Known kidney disease history (planned start)
   - No prior history (unplanned/"crash" start) AND survives >90 days
3. Treatment continues for at least 90 days OR patient:
   - Has planned start and dies within 90 days
   - Transfers to another unit

## Timeline Example
```
Key:
X = Treatment
- = No Treatment
* = Death
T = Transfer

90 days prior                                                     90 days after
|<--------------------->|<----------------------->|<----------------->|
[Gap Check]             [Start]                   [End]           [Follow-up] 

Incident          ------|---XXXXXXXXXXXXXXXXXXXXXX|XXXXX  (Continuous after start)
Incident          XXX---|---------XXXXXXXXXXXXXXXX|XXXXX  (>90 day gap timeline resets)
Incident          CKD---|--XXXX*------------------|-----  (Dies within 90 days and prior history)
Incident          ------|---XXX--XXX-XXXXXXXXX-XXX|XXXXX  (Discontinuous with short gaps)
Incident          ------|-----------------------XX|XXXXX  (Begins at end of window and continues)
Incident          ------|---XXXXXXXXXXXXXXXXT-----|-----  (Treatment ends with transfer out)

Not Incident 
Not Incident      ------|--XXXX*------------------|-----  (Dies within 90 days and no prior history)
Not incident      XXX---|-XXXXXXXXXXXXXXXXXXXXXXXX|XXXXX  (<90 day gap and timeline begins prior)
```

## UKRDC Entities Used
- [PatientRecord](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450149/PatientRecord): ukrdcid, sendingextract
- [Patient](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450145/Patient): deathtime
- [Treatment](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450155/Treatment+Encounter): qbl05, hdp04, fromtime, totime, dischargereasoncode, healthcarefacilitycode
- [ModalityCodes](https://renalregistry.atlassian.net/l/cp/Ac1YeFfH): registry_code_type

## Data Quality & Limitations
The software used to calculate the statistics should be considered experimental and is subject to the following non-exhaustive limitations:
1. Recent Window Effects (<90 days follow-up):
   - Cannot confirm treatment continuation
   - May count some patients receiving acute treatment

2. Data Coverage:
   - Inter-unit transfers appear as new starts
   - These may inflate incidence rates

3. Data Quality:
   - UKRDC contains uncleaned data and may be incomplete or contain errors



### Total Patient Population : 456
